# BERTopic Diversity Tuning using IRBO

This notebook computes **IRBO (Inverted Rank-Biased Overlap)** diversity scores
for every hyperparameter combination already evaluated in `coherence.ipynb`.

**Workflow:**
1. Load pre-computed embeddings and datasets per subject
2. For each parameter combination in `coherence_results.csv`, retrain the BERTopic model
3. Extract topic words and compute IRBO diversity
4. Save diversity results to `diversity_results.csv`
5. Merge with coherence scores to compute **Topic Quality** (harmonic mean)
6. Save combined results to `quality_results.csv`
7. Identify and save the best model per subject by Topic Quality

**Reference:** Bianchi et al. (2021) — *"Pre-training is a Hot Topic"*

In [ ]:
import os
import gc
import itertools
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Optional, Tuple
from itertools import combinations
from tqdm import tqdm
import warnings

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

pd.set_option('display.max_colwidth', None)
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [2]:
VERSION = "v1"
LIST_SUBJECT = ["cs", "math", "physics"]

SUBJECT_MODEL = {
    "cs":      {"name": "all-distilroberta-v1",  "safe_name": "all_distilroberta_v1",    "dim": 768},
    "math":    {"name": "BAAI/bge-base-en-v1.5", "safe_name": "BAAI_bge_base_en_v1.5",   "dim": 768},
    "physics": {"name": "all-distilroberta-v1",  "safe_name": "all_distilroberta_v1",    "dim": 768},
}

PARAM_GRID = {
    "min_cluster_size": [100, 150, 200],
    "min_samples": [5, 10, 20],
    "n_neighbors": [10, 15, 25],
    "n_components": [5, 10],
}

# IRBO config (from quality.ipynb)
TOP_N_WORDS = 10
RBO_P = 0.9

# Paths
BASE_DIR = Path("../../../../data/preprocess")
EMBEDDING_DIR = Path("../../../../embedding")
TUNING_DIR = Path("../../../../results/bertopic/tuning/hdbscan")

TUNING_DIR.mkdir(parents=True, exist_ok=True)

RESULT_CSV = TUNING_DIR / "coherence_results.csv"
DIVERSITY_CSV = TUNING_DIR / "diversity_results.csv"
QUALITY_CSV = TUNING_DIR / "quality_results.csv"

total_combos = 1
for values in PARAM_GRID.values():
    total_combos *= len(values)

print("Model per subject:")
for subj, info in SUBJECT_MODEL.items():
    print(f"  {subj}: {info['name']}")
print(f"\nGrid search: {total_combos} combinations per subject")
print(f"Total trials: {total_combos * len(LIST_SUBJECT)}")
print(f"\nCoherence results: {RESULT_CSV}")
print(f"Diversity results: {DIVERSITY_CSV}")
print(f"Quality results:   {QUALITY_CSV}")

Model per subject:
  cs: all-distilroberta-v1
  math: BAAI/bge-base-en-v1.5
  physics: all-distilroberta-v1

Grid search: 54 combinations per subject
Total trials: 162

Coherence results: ../../../../results/bertopic/tuning/hdbscan/coherence_results.csv
Diversity results: ../../../../results/bertopic/tuning/hdbscan/diversity_results.csv
Quality results:   ../../../../results/bertopic/tuning/hdbscan/quality_results.csv


In [ ]:
def load_dataset(subject: str) -> pd.DataFrame:
    file_path = BASE_DIR / subject / "emb" / f"{VERSION}.csv"
    if not file_path.exists():
        print(f"File not found: {file_path}")
        return None
    return pd.read_csv(file_path)


def load_mmap_embeddings(
    mmap_path: str,
    num_documents: int,
    embedding_dim: int,
    dtype: str = "float32"
) -> Optional[np.memmap]:
    try:
        return np.memmap(
            mmap_path, dtype=dtype, mode="r",
            shape=(num_documents, embedding_dim)
        )
    except Exception as e:
        print(f"Error loading embeddings: {e}")
        return None


def get_topic_words(topic_model: BERTopic, top_n: int = 10):
    """Extract top-N words for each topic (excluding topic -1), preserving rank order."""
    topics_words = []
    topic_ids = []
    for topic_id in topic_model.get_topics():
        if topic_id == -1:
            continue
        words = [word for word, _ in topic_model.get_topic(topic_id)[:top_n]]
        topics_words.append(words)
        topic_ids.append(topic_id)
    return topics_words, topic_ids





def rbo(list1, list2, p=0.9):
    if not list1 and not list2:
        return 1.0
    if not list1 or not list2:
        return 0.0

    # assign short (S) and long (L)
    if len(list1) <= len(list2):
        S, L = list1, list2
    else:
        S, L = list2, list1

    s, l = len(S), len(L)

    S_seen = set()
    L_seen = set()

    X = 0  # overlap
    rbo = 0.0
    disjoint = 0.0
    ext_term = 0.0

    for d in range(l):
        if d < s:
            s_item = S[d]
            S_seen.add(s_item)
        else:
            s_item = None

        l_item = L[d]
        L_seen.add(l_item)

        overlap_incr = 0

        if d < s:
            if s_item == l_item:
                overlap_incr = 1
            else:
                if s_item in L_seen:
                    overlap_incr += 1
                if l_item in S_seen:
                    overlap_incr += 1
        else:
            if l_item in S_seen:
                overlap_incr = 1

        X += overlap_incr

        if d < s:
            A_d = 2.0 * X / (len(S_seen) + len(L_seen))
        else:
            A_d = X / (d + 1)

        rbo += (1 - p) * (p ** d) * A_d

        if d < s:
            ext_term = A_d * (p ** (d + 1))
        else:
            X_s = X - overlap_incr if d == s else X_s
            disjoint += (1 - p) * (p ** d) * (
                X_s * (d + 1 - s) / ((d + 1) * s)
            )
            ext_term = (
                ((X - X_s) / (d + 1) + X_s / s)
                * (p ** (d + 1))
            )

        # optional optimization (safe)
        if p ** d < 1e-12:
            break

    return min(max(rbo + disjoint + ext_term, 0.0), 1.0)

def calculate_irbo(topics_words, p=0.9):
    """
    Calculate mean IRBO (Inverted RBO) diversity across all topic pairs.
    Returns mean_irbo in [0, 1]. Higher = more diverse.
    """
    if len(topics_words) < 2:
        return 0.0
    
    irbo_scores = []
    for (i, j) in combinations(range(len(topics_words)), 2):
        similarity = rbo(topics_words[i], topics_words[j], p=p)
        irbo_scores.append(1.0 - similarity)
    
    return np.mean(irbo_scores)

## Load Datasets & Embeddings

In [4]:
subject_data = {}

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"Loading {subject.upper()}...")
    
    # Load dataset
    df = load_dataset(subject)
    if df is None:
        continue
    print(f"  Documents: {len(df):,}")
    
    # Get docs
    docs = df["text"].fillna("").tolist()
    
    # Load embeddings
    model_info = SUBJECT_MODEL[subject]
    mmap_path = EMBEDDING_DIR / subject / f"{model_info['safe_name']}_{VERSION}.mmap"
    embeddings = load_mmap_embeddings(
        str(mmap_path), len(df), model_info["dim"]
    )
    if embeddings is None:
        continue
    print(f"  Embeddings: {embeddings.shape} (model: {model_info['name']})")
    
    subject_data[subject] = {
        "docs": docs,
        "embeddings": embeddings,
    }

print(f"\n{'='*60}")
print(f"Loaded {len(subject_data)} subjects: {list(subject_data.keys())}")


Loading CS...
  Documents: 165,756
  Embeddings: (165756, 768) (model: all-distilroberta-v1)

Loading MATH...
  Documents: 157,085
  Embeddings: (157085, 768) (model: BAAI/bge-base-en-v1.5)

Loading PHYSICS...
  Documents: 146,311
  Embeddings: (146311, 768) (model: all-distilroberta-v1)

Loaded 3 subjects: ['cs', 'math', 'physics']


## Load Coherence Results

In [5]:
coherence_df = pd.read_csv(RESULT_CSV)
print(f"Coherence results loaded: {len(coherence_df)} rows")
print(f"Subjects: {coherence_df['subject'].unique().tolist()}")
print(f"\nColumns: {coherence_df.columns.tolist()}")
coherence_df.head()

Coherence results loaded: 162 rows
Subjects: ['cs', 'physics', 'math']

Columns: ['subject', 'model', 'min_cluster_size', 'min_samples', 'n_neighbors', 'n_components', 'n_topics', 'coherence', 'outlier_ratio']


,subject,model,min_cluster_size,min_samples,n_neighbors,n_components,n_topics,coherence,outlier_ratio
0,cs,all-distilroberta-v1,100,5,10,5,298,0.728101,0.333719
1,cs,all-distilroberta-v1,100,5,10,10,301,0.727900,0.346196
2,cs,all-distilroberta-v1,100,5,15,5,292,0.730295,0.376210
3,cs,all-distilroberta-v1,100,5,15,10,288,0.730753,0.381959
4,cs,all-distilroberta-v1,100,5,25,5,270,0.728918,0.399901


## Grid Search — IRBO Diversity

For each parameter combination in the coherence results, retrain the BERTopic model
and compute IRBO diversity. **Resume support** is built in — if `diversity_results.csv`
exists, already-computed combinations are skipped.

In [6]:
# Load existing results for resume support
if DIVERSITY_CSV.exists():
    existing_df = pd.reasrc/bertopic/modeling/quality.ipynbd_csv(DIVERSITY_CSV)
    diversity_results = existing_df.to_dict('records')
    completed = set()
    for _, row in existing_df.iterrows():
        key = (row['subject'], int(row['min_cluster_size']), int(row['min_samples']),
               int(row['n_neighbors']), int(row['n_components']))
        completed.add(key)
    print(f"Resuming: {len(completed)} combinations already completed")
else:
    diversity_results = []
    completed = set()
    print("Starting fresh")

total = len(coherence_df)
remaining = total - len(completed)
print(f"Total combinations: {total}")
print(f"Remaining: {remaining}")

for idx, row in tqdm(coherence_df.iterrows(), total=total, desc="Computing IRBO"):
    subject = row['subject']
    mcs = int(row['min_cluster_size'])
    ms = int(row['min_samples'])
    nn = int(row['n_neighbors'])
    nc = int(row['n_components'])
    
    key = (subject, mcs, ms, nn, nc)
    
    # Skip if already computed
    if key in completed:
        continue
    
    if subject not in subject_data:
        print(f"  ⚠ Skipping {subject}: data not loaded")
        continue
    
    docs = subject_data[subject]["docs"]
    embeddings = subject_data[subject]["embeddings"]
    
    try:
        # Create UMAP + HDBSCAN with same params
        umap_model = UMAP(
            n_neighbors=nn,
            n_components=nc,
            min_dist=0.0,
            metric='cosine',
            random_state=42,
        )
        hdbscan_model = HDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms,
            metric='euclidean',
            prediction_data=True,
        )
        
        # Train BERTopic (no embedding model needed — using pre-computed)
        topic_model = BERTopic(
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            calculate_probabilities=False,
            verbose=False,
        )
        topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)
        
        # Count topics (excluding -1)
        n_topics = len(topic_model.get_topic_info()) - 1
        
        # Compute IRBO
        topics_words, topic_ids = get_topic_words(topic_model, top_n=TOP_N_WORDS)
        irbo_mean = calculate_irbo(topics_words, p=RBO_P)
        
        result = {
            "subject": subject,
            "min_cluster_size": mcs,
            "min_samples": ms,
            "n_neighbors": nn,
            "n_components": nc,
            "n_topics": n_topics,
            "irbo_mean": irbo_mean,
        }
        
        diversity_results.append(result)
        completed.add(key)
        
        # Incremental save
        pd.DataFrame(diversity_results).to_csv(DIVERSITY_CSV, index=False)
        
        del topic_model, umap_model, hdbscan_model
        gc.collect()
        
    except Exception as e:
        print(f"  ✗ Error [{subject} mcs={mcs} ms={ms} nn={nn} nc={nc}]: {e}")
        continue

print(f"\n{'='*60}")
print(f"IRBO diversity computation complete!")
print(f"Results saved to: {DIVERSITY_CSV}")
print(f"Total results: {len(diversity_results)}")

Starting fresh
Total combinations: 162
Remaining: 162


Computing IRBO: 100%|██████████| 162/162 [5:54:53<00:00, 131.44s/it]  


IRBO diversity computation complete!
Results saved to: ../../../../results/bertopic/tuning/hdbscan/diversity_results.csv
Total results: 162


## Merge with Coherence & Compute Topic Quality

$$\text{Topic Quality} = 2 \times \frac{\text{Coherence} \times \text{IRBO}}{\text{Coherence} + \text{IRBO}}$$

In [7]:
# Load results
coherence_df = pd.read_csv(RESULT_CSV)
diversity_df = pd.read_csv(DIVERSITY_CSV)

print(f"Coherence results: {len(coherence_df)} rows")
print(f"Diversity results: {len(diversity_df)} rows")

# Merge on parameter columns
merge_cols = ['subject', 'min_cluster_size', 'min_samples', 'n_neighbors', 'n_components']
combined_df = coherence_df.merge(
    diversity_df[merge_cols + ['irbo_mean']],
    on=merge_cols,
    how='inner'
)

# Topic Quality = harmonic mean of coherence and IRBO
combined_df['topic_quality'] = (
    2 * combined_df['coherence'] * combined_df['irbo_mean'] /
    (combined_df['coherence'] + combined_df['irbo_mean'])
)

# Save
combined_df.to_csv(QUALITY_CSV, index=False)

print(f"\nCombined results: {len(combined_df)} rows")
print(f"Saved to: {QUALITY_CSV}")

Coherence results: 162 rows
Diversity results: 162 rows

Combined results: 162 rows
Saved to: ../../../../results/bertopic/tuning/hdbscan/quality_results.csv


## Quality Results Table

In [8]:
print("\n" + "=" * 130)
print("COMBINED TOPIC QUALITY RESULTS (Coherence × IRBO)")
print("=" * 130)
print(
    f"{'Subject':<10} "
    f"{'MCS':>5} {'MS':>5} {'NN':>5} {'NC':>5} "
    f"{'#Topics':>8} {'Coherence':>10} {'IRBO':>8} {'Quality':>8}"
)
print("-" * 130)

for subject in LIST_SUBJECT:
    subject_data_df = combined_df[combined_df['subject'] == subject].sort_values(
        'topic_quality', ascending=False
    )
    for _, row in subject_data_df.iterrows():
        print(
            f"{row['subject']:<10} "
            f"{int(row['min_cluster_size']):>5} "
            f"{int(row['min_samples']):>5} "
            f"{int(row['n_neighbors']):>5} "
            f"{int(row['n_components']):>5} "
            f"{int(row['n_topics']):>8} "
            f"{row['coherence']:>10.4f} "
            f"{row['irbo_mean']:>8.6f} "
            f"{row['topic_quality']:>8.4f}"
        )
    print("-" * 130)

print("=" * 130)


COMBINED TOPIC QUALITY RESULTS (Coherence × IRBO)
Subject      MCS    MS    NN    NC  #Topics  Coherence     IRBO  Quality
----------------------------------------------------------------------------------------------------------------------------------
cs           100    10    25     5      261     0.7364 0.995051   0.8464
cs           150     5    15     5      202     0.7352 0.994552   0.8454
cs           100    20    25     5      243     0.7345 0.994984   0.8452
cs           100     5    25    10      256     0.7340 0.994728   0.8447
cs           100    10    15     5      273     0.7332 0.994899   0.8442
cs           150    20    10     5      208     0.7329 0.994433   0.8439
cs           200     5    15     5      174     0.7327 0.993979   0.8436
cs           150    10    25     5      187     0.7326 0.994137   0.8435
cs           150     5    25     5      197     0.7322 0.994485   0.8434
cs           150    20    25     5      184     0.7321 0.994403   0.8433
cs           15

## Best Model per Subject & Save

In [9]:
print("\n" + "=" * 100)
print("BEST MODEL PER SUBJECT (by Topic Quality)")
print("=" * 100)

best_models = {}

for subject in LIST_SUBJECT:
    subject_df = combined_df[combined_df['subject'] == subject]
    if len(subject_df) == 0 or subject_df['topic_quality'].isna().all():
        print(f"\n  {subject.upper()}: No results")
        continue
    
    best = subject_df.loc[subject_df['topic_quality'].idxmax()]
    best_models[subject] = best
    
    print(f"\n  {subject.upper()}:")
    print(f"    Params:     mcs={int(best['min_cluster_size'])}, ms={int(best['min_samples'])}, "
          f"nn={int(best['n_neighbors'])}, nc={int(best['n_components'])}")
    print(f"    Coherence:  {best['coherence']:.4f}")
    print(f"    IRBO Mean:  {best['irbo_mean']:.6f}")
    print(f"    Quality:    {best['topic_quality']:.4f}")
    print(f"    # Topics:   {int(best['n_topics'])}")

print("\n" + "=" * 100)

# Overall best
if len(combined_df) > 0:
    best_overall = combined_df.loc[combined_df['topic_quality'].idxmax()]
    print(f"\n★ OVERALL BEST: {best_overall['subject']} "
          f"(mcs={int(best_overall['min_cluster_size'])}, ms={int(best_overall['min_samples'])}, "
          f"nn={int(best_overall['n_neighbors'])}, nc={int(best_overall['n_components'])}) "
          f"with Quality = {best_overall['topic_quality']:.4f}")
    print(f"  (Coherence={best_overall['coherence']:.4f}, IRBO={best_overall['irbo_mean']:.6f})")


BEST MODEL PER SUBJECT (by Topic Quality)

  CS:
    Params:     mcs=100, ms=10, nn=25, nc=5
    Coherence:  0.7364
    IRBO Mean:  0.995051
    Quality:    0.8464
    # Topics:   261

  MATH:
    Params:     mcs=150, ms=10, nn=15, nc=5
    Coherence:  0.7229
    IRBO Mean:  0.993421
    Quality:    0.8369
    # Topics:   150

  PHYSICS:
    Params:     mcs=100, ms=5, nn=25, nc=10
    Coherence:  0.7466
    IRBO Mean:  0.995314
    Quality:    0.8532
    # Topics:   232


★ OVERALL BEST: physics (mcs=100, ms=5, nn=25, nc=10) with Quality = 0.8532
  (Coherence=0.7466, IRBO=0.995314)


## Retrain & Save Best Models

In [10]:
for subject, best in best_models.items():
    print(f"\n{'='*60}")
    print(f"Retraining best model for {subject.upper()}...")
    
    if subject not in subject_data:
        print(f"  ⚠ Skipping: data not loaded")
        continue
    
    docs = subject_data[subject]["docs"]
    embeddings = subject_data[subject]["embeddings"]
    
    mcs = int(best['min_cluster_size'])
    ms = int(best['min_samples'])
    nn = int(best['n_neighbors'])
    nc = int(best['n_components'])
    
    umap_model = UMAP(
        n_neighbors=nn,
        n_components=nc,
        min_dist=0.0,
        metric='cosine',
        random_state=42,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=mcs,
        min_samples=ms,
        metric='euclidean',
        prediction_data=True,
    )
    
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        calculate_probabilities=False,
        verbose=False,
    )
    topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)
    
    n_topics = len(topic_model.get_topic_info()) - 1
    print(f"  Topics: {n_topics}")
    print(f"  Params: mcs={mcs}, ms={ms}, nn={nn}, nc={nc}")
    
    # Save
    save_path = TUNING_DIR / f"best_{subject}_quality"
    topic_model.save(str(save_path))
    print(f"  Saved to: {save_path}")
    
    del topic_model, umap_model, hdbscan_model
    gc.collect()

print(f"\n{'='*60}")
print("All best models saved!")


Retraining best model for CS...


2026-04-11 12:52:44,890 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


  Topics: 258
  Params: mcs=100, ms=10, nn=25, nc=5
  Saved to: ../../../../results/bertopic/tuning/hdbscan/best_cs_quality

Retraining best model for MATH...


2026-04-11 12:55:00,084 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


  Topics: 150
  Params: mcs=150, ms=10, nn=15, nc=5
  Saved to: ../../../../results/bertopic/tuning/hdbscan/best_math_quality

Retraining best model for PHYSICS...


2026-04-11 12:57:39,323 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


  Topics: 233
  Params: mcs=100, ms=5, nn=25, nc=10
  Saved to: ../../../../results/bertopic/tuning/hdbscan/best_physics_quality

All best models saved!
